# Cycle Benchmarking (Cycle B)

Estimate the process fidelity of a **dressed cycle** `G̃R̃` and the per-cycle error
`r = 1 − F_RC`, SPAM-robustly — resolving the direction-dependent Pauli fidelities `λ_P`
that Standard RB averages away.

Spec & cell plan: see [`CRB.md`](CRB.md).

In [1]:
# ── Cell 1 — Imports ──────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import itertools

# Quantum info: Paulis, Cliffords, channel representations
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import (
    Pauli, PauliList, Clifford, Operator,
    SuperOp, PTM, DensityMatrix, random_pauli,
)

# Aer: simulator, noise channels, sampler primitive (hardware-portable path)
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel, QuantumError, ReadoutError,
    depolarizing_error, pauli_error, coherent_unitary_error,
)
from qiskit_aer.primitives import SamplerV2 as AerSampler
rng = np.random.default_rng(42)

In [2]:
# Cell 2 - Helpers
# make_cycle
def make_cycle(gate_name, n_qubits):
    # return the QC with the gate appended
    qc = QuantumCircuit(n_qubits)
    if gate_name == "I":
        order = 1
    elif gate_name == "s":
        order = 4
        for qbit in range(n_qubits):
            qc.s(qbit)
    elif gate_name == "cz":
        order = 2
        qc.cz(0, 1)
    elif gate_name == "H":
        qc.h(0)
        order = 2
    
    return qc, order

def make_noise(noise, n_qubits, label):
    kind = noise["kind"]
    if kind == "depolarizing":
        error = depolarizing_error(noise["p"], n_qubits)
    elif kind == "pauli":
        error = pauli_error(list(noise["probs"].items()))

    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(error, label)
    return nm, error.to_quantumchannel()  

# create the pauli comnibations based on qubit size
# for 1 qubit = 4 paulis and 2 qubit =15 paulis use all of them as K
# for 3 qubits onwards trim the list to get a smaller K then len(paulis)

def make_paulis(n_qubits, rng, K=None):

    paulis = [''.join(p) for p in itertools.product("IXYZ", repeat = n_qubits)]
    paulis.remove("I"*n_qubits) # remove the identity
    if n_qubits>2 and K is not None and K<len(paulis):
        return [paulis[i] for i in rng.choice(len(paulis), size=K, replace=False)]
    else:
        return paulis # return as is do not trim for 1 and 2 qubits
    

In [3]:
# calculate expected lambdas for different noise models
'''noises = {
    "depolarizing": {"kind":"depolarizing", "p":0.05},
    "dephasing":    {"kind":"pauli", "probs":{"I":0.95, "Z":0.05}},
    "bit_flip":     {"kind":"pauli", "probs":{"I":0.95, "X":0.05}},
    "biased":       {"kind":"pauli", "probs":{"I":0.90, "X":0.03, "Y":0.02, "Z":0.05}},
}'''

# helpers for expected lambdas here

def to_probs(noise):
    if noise["kind"] == "depolarizing": 
        # see above depol doesnt have probs
        p = noise["p"]
        return {"I": 1-3*p/4, "X": p/4, "Y": p/4, "Z": p/4}
    return noise["probs"]

def expected_lambda(probs, P):
    P = Pauli(P)
    return sum(prob * (1 if P.commutes(Pauli(Q)) else -1) for Q, prob in probs.items())

In [4]:
# build the ideal cycle C(P) and the real_noisy_cycle C̃(ρ) for a given Pauli P, sequence length m and random sequence index l

def calculate_ideal_circuit(P, qc):
    output = Pauli(P).evolve(qc, frame='s')
    assert output.phase in (0,2)
    return 1 if output.phase == 0 else -1
    

def prep_eigenstate(qc, P):
    for qbit, op in enumerate(P[::-1]): # we pass pauli as XZ but qiskit thinks Z is first qubit ie q0 so we reverse this by [::-1]
        if op == "X":
            qc.h(qbit)
        elif op == "Y":
            qc.h(qbit)
            qc.s(qbit)
    
def apply_dressed(qc, random_paulis, G):
    for i , random_pauli in enumerate(random_paulis):
        for qbit, op in enumerate(random_pauli[::-1]):
            if op == "X":
                qc.x(qbit)
            elif op == "Y":
                qc.y(qbit)
            elif op == "Z":
                qc.z(qbit)
        if i < len(random_paulis)-1:
            qc.append(G, range(qc.num_qubits))


def measure_P(qc, P):
    for qbit, op in enumerate(P[::-1]):
        if op == "X":
            qc.h(qbit)
        elif op == "Y":
            qc.sdg(qbit)
            qc.h(qbit)
    qc.measure(range(qc.num_qubits), range(qc.num_qubits))


def build_real_circuit(P, m, random_paulis, n_qubits, G_gate):
    qc = QuantumCircuit(n_qubits, n_qubits)
    prep_eigenstate(qc, P)
    apply_dressed(qc, random_paulis, G_gate)
    measure_P(qc, P)
    return qc


def run_once(P, m, n_qubits, shots, noise_model, G_map, G_gate, rng):
    # this f_pml = Tr[C(P), C̃(ρ)] is the core of the protocol: one random sequence, one shot at the overlap
    random_paulis = ["".join(rng.choice(list("IXYZ"), n_qubits)) for _ in range(m+1)]

    run_qc = build_real_circuit(P, m, random_paulis, n_qubits, G_gate)

    ideal_qc = QuantumCircuit(n_qubits)
    apply_dressed(ideal_qc, random_paulis, G_map) # check this
    pauli_sign = calculate_ideal_circuit(P, ideal_qc)

    counts = AerSimulator(noise_model=noise_model).run(run_qc, shots=shots).result().get_counts()
    expectation_value = get_expectation_value(counts, P, shots)
    return pauli_sign * expectation_value

def get_expectation_value(counts, P, shots):
    P = P[::-1] # reverse to match qiskit's ordering
    total = 0
    for bitstring, c in counts.items():
        bitstring = bitstring[::-1] # reverse to match qiskit's ordering
        eigenvalue = 1
        for i in range(len(P)):
            if P[i] !="I" and bitstring[i] =='1':
                eigenvalue *= -1
        total += eigenvalue * c
    return total / shots

'''P has only ±1 eigenvalues. The loop's whole job is: for each measured bitstring, decide whether it's the +1 outcome or the −1 outcome — then average them. ⟨P⟩ is the average of those ±1's, so a number in [−1, +1].

1. For 1 qubit it's trivial (P="Z"): bit "0" → +1, bit "1" → −1. With counts={"0":95,"1":5}:

<Z> = (95·(+1) + 5·(−1)) / 100 = 0.9

That's all. The loop for n=1 literally does (-1)^0 = +1 for "0" and (-1)^1 = −1 for "1".

2. For multiple qubits, a single bitstring's eigenvalue isn't one bit — it's the product of each qubit's ±1. After measure_P rotates P→Z…Z, each qubit gives +1 (bit 0) or −1 (bit 1), and P's eigenvalue is their product:

eigenvalue = (+1 or −1) × (+1 or −1) × ...   over P's support
           = (−1)^(number of 1-bits on the support)
    
'''

def run_cb(paulis_K, m1, m2, L, n_qubits, shots, noise_model, G_map, G_gate, rng):
    lambdas = {}
    for P in paulis_K:
        F = {}
        for m in (m1, m2):
            F[m] = np.mean([run_once(P, m, n_qubits, shots, noise_model, G_map, G_gate, rng) for _ in range(L)])
        lambdas[P] = (F[m2]/F[m1]) ** (1/(m2-m1))
    return lambdas

def print_lambdas(name, measured, expected=None):
    print(f"\n  {name}")
    if expected is None:
        print(f"  {'P':<5}{'measured':>10}")
        for P in measured:
            print(f"  {P:<5}{measured[P]:>10.4f}")
        print(f"  {'F_RC':<5}{np.mean(list(measured.values())):>10.4f}")
    else:    
        print(f"  {'P':<5}{'expected':>10}{'measured':>10}{'Δ':>9}")
        for P in expected:
            print(f"  {P:<5}{expected[P]:>10.4f}{measured[P]:>10.4f}{measured[P]-expected[P]:>+9.4f}")
        e, m = np.mean(list(expected.values())), np.mean(list(measured.values()))
        print(f"  {'F_RC':<5}{e:>10.4f}{m:>10.4f}{m-e:>+9.4f}")


In [5]:
# ── Cell 2 — Config (single source of truth) ──────────────────────────
# Terminology used throughout the notebook:
#   ideal_cycle = C(P)    : ideal expected outcome, a signed Pauli ±P   (classical, always exact)
#   real_cycle  = C̃(ρ)    : noisy output from running the SAME circuit  (sim or hardware)
#   overlap   f_{P,m,l} = Tr[ ideal_cycle · real_cycle ]                (= +1 if noiseless)

# --- System ---
n_qubits   = 1 # 1,2 to validate; up to ~10 for the scaling demo (cell 14)
cycle_gate = "I" # "identity" | "s" (1q layer) | "cz" (2q entangler) — the cycle G under test

# --- Protocol: TWO sequence lengths (paper Eq.7), each with G^m = I ---
m1 = 4  # first length  (paper: m1 = 4)
m2 = 12 # second length (multiple of 4 -> G^m = I; 8 also valid)
#   estimator:  lambda_P = (F_{P,m2} / F_{P,m1}) ** (1 / (m2 - m1))

K = 30 # # Paulis P sampled   -> F_RC = (1/K) sum_P lambda_P
#ALL non-identity if N<=2, else random subset of size K
L = 30 # # random sequences per (P,m) -> the twirl: F_{P,m} = (1/L) sum_l f_{P,m,l}
shots = 100 # # measurement repeats per circuit -> one f_{P,m,l} (paper p.4)

# --- Noise on G. CB measures lambda_P = survival of P-component:  Lambda(P) = lambda_P * P.
#"Channels preserve I" is the statement λ_I = 1 — and that's automatic, regardless of p_I: λ_I = p_I + p_X + p_Y + p_Z = (1−q) + q = 1 
#
# lambda_P = P(errors COMMUTING with P) - P(errors ANTICOMMUTING with P)
# commute: X~I,X | Y~I,Y | Z~I,Z ; each anticommutes with the other two)
# lambda_I = 1 <- always; I never decays, so not measured
#       lambda_X = (pI+pX) - (pY+pZ)
#       lambda_Y = (pI+pY) - (pX+pZ)
#       lambda_Z = (pI+pZ) - (pX+pY)
# 
# expected CB output (verify against this): order is (λ_X, λ_Y, λ_Z)
# interestingly e.g Z damages X and Y and Z is left alone e.g. dephasing
#   depolarizing p {"kind":"depolarizing","p":p} -> (1-p, 1-p, 1-p)
#   dephasing-q {"kind":"pauli","probs":{"I":1-q,"Z":q}} -> (1-2q,  1-2q,  1)
#   bit-flip-q {"kind":"pauli","probs":{"I":1-q,"X":q}} -> (1,1-2q,  1-2q)
#   biased {"I":.90,"X":.03,"Y":.02,"Z":.05} -> (0.86,  0.84,  0.90)
CYCLE_LABEL = "CB_Cycle"
noises = {
    "depolarizing": {"kind":"depolarizing", "p":0.05},
    "dephasing":    {"kind":"pauli", "probs":{"I":0.95, "Z":0.05}},
    "bit_flip":     {"kind":"pauli", "probs":{"I":0.95, "X":0.05}},
    "biased":       {"kind":"pauli", "probs":{"I":0.90, "X":0.03, "Y":0.02, "Z":0.05}},
}

for cycle_gate in ["I", "H"]:
    G_map, order = make_cycle(cycle_gate, n_qubits)
    G_gate = UnitaryGate(Operator(G_map).data, label=CYCLE_LABEL)
    print("==================================================================")
    print(f"Cycle {cycle_gate} (order {order}), {n_qubits} qubit — expected vs measured λ_P")
    # run the cycle benckhmarking 
    for name, noise in noises.items():
        # expected results
        probs = to_probs(noise)
        expected_lambdas = {P: expected_lambda(probs, P) for P in make_paulis(n_qubits, rng, K)} if cycle_gate=="I" else None

        # now the real circuit run with noise
        nm, channel = make_noise(noise, n_qubits, CYCLE_LABEL)
        measured_lambdas = run_cb(make_paulis(n_qubits, rng, K), m1, m2, L, n_qubits, shots, nm, G_map, G_gate, rng)
        print_lambdas(name, measured_lambdas, expected_lambdas)


Cycle I (order 1), 1 qubit — expected vs measured λ_P

  depolarizing
  P      expected  measured        Δ
  X        0.9500    0.9499  -0.0001
  Y        0.9500    0.9452  -0.0048
  Z        0.9500    0.9505  +0.0005
  F_RC     0.9500    0.9485  -0.0015

  dephasing
  P      expected  measured        Δ
  X        0.9000    0.9050  +0.0050
  Y        0.9000    0.8853  -0.0147
  Z        1.0000    1.0000  +0.0000
  F_RC     0.9333    0.9301  -0.0032

  bit_flip
  P      expected  measured        Δ
  X        1.0000    1.0000  +0.0000
  Y        0.9000    0.9055  +0.0055
  Z        0.9000    0.9030  +0.0030
  F_RC     0.9333    0.9362  +0.0028

  biased
  P      expected  measured        Δ
  X        0.8600    0.8859  +0.0259
  Y        0.8400    0.8534  +0.0134
  Z        0.9000    0.8966  -0.0034
  F_RC     0.8667    0.8786  +0.0120
Cycle H (order 2), 1 qubit — expected vs measured λ_P

  depolarizing
  P      measured
  X        0.9488
  Y        0.9428
  Z        0.9459
  F_RC     0.